[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_02_data_structures_algorithms/ch08_tuples_sets_dicts/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 8 Walkthrough: Tuples, Sets, and Dictionaries — Structured & Hash-Based Thinking

**Book Reference:** Chapter 8, Sections 8.1 – 8.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections.

## Section 8.3.1 — Tuples, Immutability, and Clean Unpacking

In [ ]:
clinic_gps = (23.8103, 90.4125)
patient_vitals = ("UHC-0042", 38.4, 10.2)

pid, temp_c, hb_g_dl = patient_vitals
print(f"Patient {pid}: Temp={temp_c} C, Hb={hb_g_dl} g/dL")
print("Clinic GPS:", clinic_gps, "| Type:", type(clinic_gps))


## Section 8.3.2 — Sets & Cohort Operations (`&`, `-`, `|`, $O(1)$ Membership)

In [ ]:
baseline_raw = ["P01", "P02", "P03", "P04", "P02"]
followup_raw = ["P02", "P04", "P05"]

base_set = set(baseline_raw)
foll_set = set(followup_raw)

retained = base_set & foll_set
lost_to_followup = base_set - foll_set
new_enrollees = foll_set - base_set

print("Unique baseline:", sorted(base_set))
print("Retained:       ", sorted(retained))
print("Lost:           ", sorted(lost_to_followup))
print("New enrollees:  ", sorted(new_enrollees))
print("Is P02 retained?", "P02" in retained)


## Section 8.3.3 — Dictionaries & Safe Lookups with `.get()`

In [ ]:
vaccine_stock = {"BCG": 140, "OPV": 85, "Measles": 0}

# Safe read-only lookup with fallback default 0
bcg_doses = vaccine_stock.get("BCG", 0)
rota_doses = vaccine_stock.get("Rota", 0)

print("BCG doses: ", bcg_doses)
print("Rota doses:", rota_doses)
print("Keys unchanged:", list(vaccine_stock.keys()))


## Section 8.3.4 — Core Hash-Map Idioms: Frequency Counting & Grouping

In [ ]:
triage_log = ["Routine", "Urgent", "Routine", "Moderate", "Routine"]
triage_counts = {}
for level in triage_log:
    triage_counts[level] = triage_counts.get(level, 0) + 1

intake_batch = [
    {"patient_id": "UHC-01", "district": "Dhaka"},
    {"patient_id": "UHC-02", "district": "Sylhet"},
    {"patient_id": "UHC-03", "district": "Dhaka"},
]
by_district = {}
for rec in intake_batch:
    dist = rec["district"]
    by_district.setdefault(dist, []).append(rec["patient_id"])

print("Triage counts:", triage_counts)
print("Grouped IDs:  ", by_district)


## Section 8.3.5 — Dictionary Comprehensions & Guarded Rate Mapping

In [ ]:
cases = {"Dhaka": 120, "Sylhet": 45, "Unknown": 10}
pop = {"Dhaka": 300000, "Sylhet": 150000, "Unknown": 0}

rates = {
    dist: round((cnt / pop[dist]) * 100000, 2)
    for dist, cnt in cases.items()
    if pop.get(dist, 0) > 0
}
print("Incidence per 100k:", rates)


## Section 8.3.6 — Live Preview: Grouping `clinic_triage` with `sharukh_tutor`

In [ ]:
# @title Load and Group Live clinic_triage Records (120 Patients)
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import sharukh_tutor as tutor

clinic_rows = tutor.load_data("clinic_triage")
district_groups = {}
triage_freq = {}
for row in clinic_rows:
    dist = row["district"]
    lvl = row["triage_priority"]
    district_groups.setdefault(dist, []).append(row["patient_id"])
    triage_freq[lvl] = triage_freq.get(lvl, 0) + 1

print("Patients per district:")
for dist in sorted(district_groups):
    print(f"  {dist:11s}: {len(district_groups[dist]):2d} patients")
print("Triage breakdown:", triage_freq)


## Section 8.3.7 — Break It On Purpose! (Tuple, Dict & Hashability Error Anatomy)

In [ ]:
# Intentional Bug #1: TypeError — trying to mutate an immutable tuple
clinic_gps = (23.8103, 90.4125)
clinic_gps[0] = 24.0


In [ ]:
# Intentional Bug #2: KeyError — direct [] lookup on a missing dict key
vaccine_stock = {"BCG": 140, "OPV": 85}
print(vaccine_stock["Rota"])


In [ ]:
# Intentional Bug #3: TypeError — mutable list used as a dict key / set item
bad_coords = {[23.81, 90.41]: "Dhaka Clinic"}


## Section 8.4 — Worked Problem-Structuring Blueprint: Cohort Reconciliation & District Anemia Audit

In [ ]:
def reconcile_and_audit(baseline_ids, followup_ids, hb_records):
    """Audit cohort retention via sets and district anemia via dicts."""
    base_set = set(baseline_ids)
    foll_set = set(followup_ids)
    retained = base_set & foll_set
    lost = base_set - foll_set

    valid_by_dist = {}
    anemic_by_dist = {}
    for rec in hb_records:
        hb = rec["hb_g_dl"]
        if hb is None or hb == 999:
            continue
        dist = rec["district"]
        valid_by_dist[dist] = valid_by_dist.get(dist, 0) + 1
        if hb < 11.0:
            anemic_by_dist[dist] = anemic_by_dist.get(dist, 0) + 1

    prevalence = {
        dist: round((anemic_by_dist.get(dist, 0) / n) * 100, 1)
        for dist, n in valid_by_dist.items()
    }
    return (len(retained), len(lost), prevalence)

sample_records = [
    {"district": "Dhaka", "hb_g_dl": 10.4},
    {"district": "Dhaka", "hb_g_dl": 11.6},
    {"district": "Dhaka", "hb_g_dl": None},
    {"district": "Sylhet", "hb_g_dl": 9.8},
    {"district": "Sylhet", "hb_g_dl": 999},
]
n_ret, n_lost, prev_map = reconcile_and_audit(
    ["M01", "M02", "M03", "M02"],
    ["M02", "M03", "M04"],
    sample_records,
)
print(f"Retained: {n_ret} | Lost: {n_lost}")
print("District anemia (%):", prev_map)
